In [1]:
import pandas as pd
import numpy as np
from pathlib import Path 

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

In [2]:
PATH_TO = Path("/Volumes/Extreme SSD/MIMIC")

# ── MIMIC-IV-ED tables ──
edstays = pd.read_csv(PATH_TO / "mimic-iv-ed-2.2/ed/edstays.csv" )     
triage  = pd.read_csv(PATH_TO / "mimic-iv-ed-2.2/ed/triage.csv")
medrecon = pd.read_csv(PATH_TO / "mimic-iv-ed-2.2/ed/medrecon.csv")

# ── MIMIC-IV hosp tables ──
patients   = pd.read_csv(PATH_TO / "mimic-iv-3.1/hosp/patients.csv")
admissions = pd.read_csv(PATH_TO / "mimic-iv-3.1/hosp/admissions.csv")

print(f"edstays:    {edstays.shape}")
print(f"triage:     {triage.shape}")
print(f"medrecon:   {medrecon.shape}")
print(f"patients:   {patients.shape}")
print(f"admissions: {admissions.shape}")


edstays:    (425087, 9)
triage:     (425087, 11)
medrecon:   (2987342, 9)
patients:   (364627, 6)
admissions: (546028, 16)


In [3]:
# 1:1 merge on stay_id — every ED stay should have a triage record
ed = edstays.merge(triage, on=["subject_id", "stay_id"], how="left")

print(f"edstays rows:       {len(edstays):,}")
print(f"after triage merge: {len(ed):,}")
print(f"triage match rate:  {ed['acuity'].notna().mean():.1%}")

edstays rows:       425,087
after triage merge: 425,087
triage match rate:  98.4%


In [4]:
# Debug: check what columns came from the patients merge
print("Columns after merge:")
print([c for c in ed.columns if "anchor" in c.lower() or "age" in c.lower()])
print("\nPatients table columns:")
print(list(patients.columns))

Columns after merge:
[]

Patients table columns:
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']


In [5]:
# anchor_age is the patient's age at anchor_year
# to get age at time of ED visit, we compute the year offset
ed = ed.merge(
    patients[["subject_id", "anchor_age", "anchor_year", "dod"]],
    on="subject_id",
    how="left",
)
ed["intime"] = pd.to_datetime(ed["intime"])
ed["ed_year"] = ed["intime"].dt.year


# compute actual age at ED visit
ed["ed_year"] = ed["intime"].dt.year
ed["age"] = ed["anchor_age"] + (ed["ed_year"] - ed["anchor_year"])

# sanity check
print(f"Age range: {ed['age'].min()} – {ed['age'].max()}")
print(f"Age missing: {ed['age'].isna().sum()}")
print(f"\nAge distribution:")
print(ed["age"].describe())

Age range: 18.0 – 103.0
Age missing: 76

Age distribution:
count    425011.000000
mean         52.864493
std          20.619906
min          18.000000
25%          35.000000
50%          53.000000
75%          69.000000
max         103.000000
Name: age, dtype: float64


In [6]:
# admissions has insurance, language, marital_status
# only available for patients who were admitted (hadm_id not null)
admissions_demo = admissions[
    ["hadm_id", "insurance", "language", "marital_status"]
].drop_duplicates(subset=["hadm_id"])

ed = ed.merge(admissions_demo, on="hadm_id", how="left")

print(f"Patients with insurance info: {ed['insurance'].notna().sum():,} / {len(ed):,}")
print(f"Patients with language info:  {ed['language'].notna().sum():,} / {len(ed):,}")
print(f"Patients with marital status: {ed['marital_status'].notna().sum():,} / {len(ed):,}")

Patients with insurance info: 199,155 / 425,087
Patients with language info:  202,874 / 425,087
Patients with marital status: 198,790 / 425,087


In [7]:
# deduplicate medrecon: a single drug can have multiple rows due to
# multiple ETC classifications. We want one row per (stay_id, drug name).
meds_deduped = medrecon.drop_duplicates(subset=["stay_id", "name"])

# aggregate into a list of drug names per stay
meds_per_stay = (
    meds_deduped
    .groupby("stay_id")["name"]
    .apply(list)
    .reset_index()
    .rename(columns={"name": "current_medications"})
)

# also get the drug class descriptions for a parallel column
classes_per_stay = (
    medrecon
    .drop_duplicates(subset=["stay_id", "etcdescription"])
    .dropna(subset=["etcdescription"])
    .groupby("stay_id")["etcdescription"]
    .apply(list)
    .reset_index()
    .rename(columns={"etcdescription": "medication_classes"})
)

ed = ed.merge(meds_per_stay, on="stay_id", how="left")
ed = ed.merge(classes_per_stay, on="stay_id", how="left")

# count of medications as a simple numeric feature
ed["n_medications"] = ed["current_medications"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print(f"Patients with medication data: {ed['current_medications'].notna().sum():,} / {len(ed):,}")
print(f"Median medications per patient: {ed['n_medications'].median():.0f}")
print(f"Max medications: {ed['n_medications'].max()}")

Patients with medication data: 307,196 / 425,087
Median medications per patient: 4
Max medications: 67


In [8]:
# pain is stored as text and may have non-numeric entries
print("Raw pain value counts (top 15):")
print(ed["pain"].value_counts().head(15))
print(f"\nTotal non-null: {ed['pain'].notna().sum():,}")

# coerce to numeric — non-numeric entries become NaN
ed["pain_raw"] = ed["pain"]  # keep original for inspection
ed["pain"] = pd.to_numeric(ed["pain"], errors="coerce")

# clip to valid range 0-10
ed.loc[ed["pain"] > 10, "pain"] = np.nan
ed.loc[ed["pain"] < 0, "pain"] = np.nan

print(f"\nAfter cleaning — pain non-null: {ed['pain'].notna().sum():,}")
print(ed["pain"].describe())

Raw pain value counts (top 15):
pain
0           140719
8            41730
10           40914
7            31423
5            29210
6            25206
9            20210
4            19721
3            16115
2            13903
13           10921
1             5903
unable        3606
Critical      1697
uta           1601
Name: count, dtype: int64

Total non-null: 412,154

After cleaning — pain non-null: 385,769
count    385769.000000
mean          4.166862
std           3.754274
min          -0.000000
25%           0.000000
50%           4.000000
75%           8.000000
max          10.000000
Name: pain, dtype: float64


In [9]:
# define plausible ranges for triage vitals
vital_ranges = {
    "temperature": (85.0, 110.0),    # °F
    "heartrate":   (20, 300),
    "resprate":    (4, 60),
    "o2sat":       (40, 100),
    "sbp":         (40, 300),
    "dbp":         (20, 200),
}

for col, (lo, hi) in vital_ranges.items():
    before = ed[col].notna().sum()
    ed.loc[(ed[col] < lo) | (ed[col] > hi), col] = np.nan
    after = ed[col].notna().sum()
    removed = before - after
    if removed > 0:
        print(f"{col}: removed {removed:,} implausible values (outside {lo}–{hi})")

print("\nVitals summary after cleaning:")
print(ed[list(vital_ranges.keys())].describe().round(1))

temperature: removed 538 implausible values (outside 85.0–110.0)
heartrate: removed 29 implausible values (outside 20–300)
resprate: removed 58 implausible values (outside 4–60)
o2sat: removed 136 implausible values (outside 40–100)
sbp: removed 203 implausible values (outside 40–300)
dbp: removed 593 implausible values (outside 20–200)

Vitals summary after cleaning:
       temperature  heartrate  resprate     o2sat       sbp       dbp
count     401134.0   407968.0  404676.0  404355.0  406593.0  405403.0
mean          98.1       85.1      17.5      98.4     135.0      77.5
std            1.0       17.7       2.3       2.1      22.3      14.7
min           85.0       20.0       4.0      42.0      40.0      20.0
25%           97.5       72.0      16.0      97.0     120.0      68.0
50%           98.0       84.0      18.0      99.0     133.0      77.0
75%           98.6       96.0      18.0     100.0     148.0      87.0
max          110.0      256.0      60.0     100.0     299.0     199.0

In [10]:
# standardize string columns
for col in ["gender", "race", "arrival_transport", "disposition"]:
    ed[col] = ed[col].str.strip().str.upper()

# inspect race categories
print("Race categories:")
print(ed["race"].value_counts())
print(f"\nArrival transport:")
print(ed["arrival_transport"].value_counts())
print(f"\nDisposition:")
print(ed["disposition"].value_counts())

Race categories:
race
WHITE                                        228123
BLACK/AFRICAN AMERICAN                        76798
OTHER                                         20752
HISPANIC/LATINO - PUERTO RICAN                14036
WHITE - OTHER EUROPEAN                         8992
HISPANIC/LATINO - DOMINICAN                    8330
BLACK/CAPE VERDEAN                             7638
ASIAN - CHINESE                                7348
ASIAN                                          7294
UNKNOWN                                        7083
WHITE - RUSSIAN                                6091
BLACK/AFRICAN                                  4887
BLACK/CARIBBEAN ISLAND                         3675
HISPANIC OR LATINO                             3141
HISPANIC/LATINO - GUATEMALAN                   2356
ASIAN - ASIAN INDIAN                           1567
ASIAN - SOUTH EAST ASIAN                       1533
HISPANIC/LATINO - SALVADORAN                   1497
WHITE - BRAZILIAN                         

In [11]:
print(f"Total rows before filtering: {len(ed):,}")

# remove rows without an acuity score (can't use as ground truth)
ed_clean = ed[ed["acuity"].notna()].copy()
print(f"After removing missing acuity: {len(ed_clean):,} (dropped {len(ed) - len(ed_clean):,})")

# remove eloped / left without being seen (incomplete encounters)
exclude_dispositions = [
    "ELOPED",
    "LEFT WITHOUT BEING SEEN",
    "LEFT AGAINST MEDICAL ADVICE",
]
before = len(ed_clean)
ed_clean = ed_clean[~ed_clean["disposition"].isin(exclude_dispositions)]
print(f"After removing eloped/LWBS/AMA: {len(ed_clean):,} (dropped {before - len(ed_clean):,})")

# cast acuity to int
ed_clean["acuity"] = ed_clean["acuity"].astype(int)

print(f"\nFinal acuity distribution:")
print(ed_clean["acuity"].value_counts().sort_index())

Total rows before filtering: 425,087
After removing missing acuity: 418,100 (dropped 6,987)
After removing eloped/LWBS/AMA: 404,461 (dropped 13,639)

Final acuity distribution:
acuity
1     23817
2    136584
3    215950
4     27108
5      1002
Name: count, dtype: int64


In [12]:
ed_clean = ed_clean.rename(columns={
    "gender":            "patient_gender",
    "race":              "patient_race",
    "arrival_transport": "arrival_mode",
    "temperature":       "triage_temperature_f",
    "heartrate":         "triage_heart_rate_bpm",
    "resprate":          "triage_resp_rate",
    "o2sat":             "triage_o2_saturation_pct",
    "sbp":               "triage_systolic_bp",
    "dbp":               "triage_diastolic_bp",
    "pain":              "triage_pain_score",
    "chiefcomplaint":    "chief_complaint",
    "n_medications":     "num_current_medications",
    "acuity":            "esi_acuity",
    "intime":            "ed_arrival_time",
    "disposition":       "ed_disposition",
    "insurance":         "insurance_type",
    "language":          "primary_language",
})

print("Renamed columns:")
print(list(ed_clean.columns))

Renamed columns:
['subject_id', 'hadm_id', 'stay_id', 'ed_arrival_time', 'outtime', 'patient_gender', 'patient_race', 'arrival_mode', 'ed_disposition', 'triage_temperature_f', 'triage_heart_rate_bpm', 'triage_resp_rate', 'triage_o2_saturation_pct', 'triage_systolic_bp', 'triage_diastolic_bp', 'triage_pain_score', 'esi_acuity', 'chief_complaint', 'anchor_age', 'anchor_year', 'dod', 'ed_year', 'age', 'insurance_type', 'primary_language', 'marital_status', 'current_medications', 'medication_classes', 'num_current_medications', 'pain_raw']


In [13]:
id_cols = ["subject_id", "hadm_id", "stay_id"]

triage_feature_cols = [
    "age",
    "patient_gender",
    "patient_race",
    "arrival_mode",
    "triage_temperature_f",
    "triage_heart_rate_bpm",
    "triage_resp_rate",
    "triage_o2_saturation_pct",
    "triage_systolic_bp",
    "triage_diastolic_bp",
    "triage_pain_score",
    "chief_complaint",
    "current_medications",
    "medication_classes",
    "num_current_medications",
]

optional_demo_cols = [
    "primary_language",
    "marital_status",
]

label_cols = ["esi_acuity"]

metadata_cols = ["ed_arrival_time", "ed_disposition"]

master_cols = id_cols + triage_feature_cols + optional_demo_cols + label_cols + metadata_cols
master = ed_clean[master_cols].copy()

print(f"Master dataframe: {master.shape[0]:,} rows × {master.shape[1]} columns")
master.head()

Master dataframe: 404,461 rows × 23 columns


,subject_id,hadm_id,stay_id,age,patient_gender,patient_race,arrival_mode,triage_temperature_f,triage_heart_rate_bpm,triage_resp_rate,triage_o2_saturation_pct,triage_systolic_bp,triage_diastolic_bp,triage_pain_score,chief_complaint,current_medications,medication_classes,num_current_medications,primary_language,marital_status,esi_acuity,ed_arrival_time,ed_disposition
0,10000032,22595853.0,33258284,52.0,F,WHITE,AMBULANCE,98.4,70.0,16.0,97.0,106.0,63.0,0.0,"Abd pain, Abdominal distention","[albuterol sulfate, emtricitabine-tenofovir [Truvada], ergocalciferol (vitam...","[Asthma/COPD Therapy - Beta 2-Adrenergic Agents, Inhaled, Short Acting, Anti...",9,English,WIDOWED,3,2180-05-06 19:17:00,ADMITTED
1,10000032,22841357.0,38112554,52.0,F,WHITE,AMBULANCE,98.9,88.0,18.0,97.0,116.0,88.0,10.0,Abdominal distention,"[albuterol sulfate, calcium carbonate, emtricitabine-tenofovir [Truvada], fu...","[Asthma/COPD Therapy - Beta 2-Adrenergic Agents, Inhaled, Short Acting, Mine...",12,English,WIDOWED,3,2180-06-26 15:54:00,ADMITTED
2,10000032,25742920.0,35968195,52.0,F,WHITE,AMBULANCE,99.4,105.0,18.0,96.0,106.0,57.0,10.0,"n/v/d, Abd pain","[emtricitabine-tenofovir [Truvada], lactulose, nicotine, raltegravir [Isentr...","[Antiretroviral - Nucleoside and Nucleotide Analog RTIs Combinations, Coloni...",7,English,WIDOWED,3,2180-08-05 20:58:00,ADMITTED
3,10000032,29079034.0,32952584,52.0,F,WHITE,AMBULANCE,97.8,87.0,14.0,97.0,71.0,43.0,7.0,Hypotension,"[albuterol sulfate, calcium carbonate, cholecalciferol (vitamin D3), emtrici...","[Asthma/COPD Therapy - Beta 2-Adrenergic Agents, Inhaled, Short Acting, Mine...",14,English,WIDOWED,2,2180-07-22 16:24:00,HOME
4,10000032,29079034.0,39399961,52.0,F,WHITE,AMBULANCE,98.7,77.0,16.0,98.0,96.0,50.0,NaN,"Abdominal distention, Abd pain, LETHAGIC","[albuterol sulfate, calcium carbonate, cholecalciferol (vitamin D3), emtrici...","[Asthma/COPD Therapy - Beta 2-Adrenergic Agents, Inhaled, Short Acting, Mine...",14,English,WIDOWED,2,2180-07-23 05:54:00,ADMITTED


In [14]:
missing = master[triage_feature_cols].isnull().mean().sort_values(ascending=False)

print("Missingness in triage features:")
print("=" * 45)
for col, pct in missing.items():
    bar = "█" * int(pct * 40)
    print(f"  {col:<25} {pct:>6.1%}  {bar}")

Missingness in triage features:
  medication_classes         26.2%  ██████████
  current_medications        25.6%  ██████████
  triage_pain_score           7.8%  ███
  triage_temperature_f        4.2%  █
  triage_o2_saturation_pct    3.4%  █
  triage_resp_rate            3.3%  █
  triage_diastolic_bp         3.1%  █
  triage_systolic_bp          2.8%  █
  triage_heart_rate_bpm       2.5%  █
  age                         0.0%  
  chief_complaint             0.0%  
  patient_gender              0.0%  
  patient_race                0.0%  
  arrival_mode                0.0%  
  num_current_medications     0.0%  


In [18]:
# save full master
master.to_csv("/Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/mimic/medical_ed_triage.csv", index=False)
print(f"Saved: /Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/mimic/medical_ed_triage.csv ({len(master):,} rows)")

# also save a quick summary
summary = {
    "total_patients": master["subject_id"].nunique(),
    "total_ed_stays": len(master),
    "acuity_distribution": master["esi_acuity"].value_counts().sort_index().to_dict(),
    "median_age": master["age"].median(),
    "median_medications": master["num_current_medications"].median(),
    "pct_admitted": master["hadm_id"].notna().mean(),
    "features": triage_feature_cols,
}

import json
print("\nDataset summary:")
print(json.dumps(summary, indent=2, default=str))

Saved: /Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/mimic/medical_ed_triage.csv (404,461 rows)

Dataset summary:
{
  "total_patients": 197469,
  "total_ed_stays": 404461,
  "acuity_distribution": {
    "1": 23817,
    "2": 136584,
    "3": 215950,
    "4": 27108,
    "5": 1002
  },
  "median_age": 54.0,
  "median_medications": 4.0,
  "pct_admitted": 0.4851098128126074,
  "features": [
    "age",
    "patient_gender",
    "patient_race",
    "arrival_mode",
    "triage_temperature_f",
    "triage_heart_rate_bpm",
    "triage_resp_rate",
    "triage_o2_saturation_pct",
    "triage_systolic_bp",
    "triage_diastolic_bp",
    "triage_pain_score",
    "chief_complaint",
    "current_medications",
    "medication_classes",
    "num_current_medications"
  ]
}


In [24]:
def select_balanced_cohort(master, n, seed=42):
    """
    Select n patients with equal representation across ESI acuity levels.
    
    - n must be divisible by 5 (one bin per acuity level)
    - No patient appears twice (deduplicated by subject_id)
    - No NaN values in any triage feature column
    
    Returns the selected subset as a DataFrame.
    """
    assert n % 5 == 0, f"n must be divisible by 5 (got {n})"
    per_bin = n // 5

    # columns that must be non-null for a fair comparison
    required_cols = [
        "age",
        "patient_gender",
        "patient_race",
        "arrival_mode",
        "triage_temperature_f",
        "triage_heart_rate_bpm",
        "triage_resp_rate",
        "triage_o2_saturation_pct",
        "triage_systolic_bp",
        "triage_diastolic_bp",
        "triage_pain_score",
        "chief_complaint",
    ]

    # drop rows with any NaN in required columns
    eligible = master.dropna(subset=required_cols).copy()
    print(f"Eligible rows (no NaN in triage features): {len(eligible):,} / {len(master):,}")

    # deduplicate by subject_id — keep the most recent visit
    eligible = (
        eligible
        .sort_values("ed_arrival_time", ascending=False)
        .drop_duplicates(subset="subject_id", keep="first")
    )
    print(f"Unique patients after dedup: {len(eligible):,}")

    # check we have enough per bin
    available = eligible["esi_acuity"].value_counts().sort_index()
    print(f"\nAvailable patients per acuity level:")
    for level, count in available.items():
        status = "OK" if count >= per_bin else "INSUFFICIENT"
        print(f"  ESI {int(level)}: {count:>6,}  (need {per_bin}) [{status}]")

    for level in range(1, 6):
        avail = available.get(level, 0)
        if avail < per_bin:
            raise ValueError(
                f"Not enough patients for ESI {level}: "
                f"need {per_bin}, have {avail}"
            )

    # sample equally from each bin
    rng = np.random.default_rng(seed)
    selected = []
    for level in range(1, 6):
        pool = eligible[eligible["esi_acuity"] == level]
        chosen = pool.sample(n=per_bin, random_state=rng.integers(1e9))
        selected.append(chosen)
        print(f"  Sampled {per_bin} from ESI {int(level)}")

    cohort = pd.concat(selected, ignore_index=True)

    # final shuffle so acuity levels aren't grouped
    cohort = cohort.sample(frac=1, random_state=rng.integers(1e9)).reset_index(drop=True)

    # sanity checks
    assert len(cohort) == n, f"Expected {n} rows, got {len(cohort)}"
    assert cohort["subject_id"].is_unique, "Duplicate patients found"
    assert cohort[required_cols].isna().sum().sum() == 0, "NaN values found"

    print(f"\nFinal cohort: {len(cohort)} patients")
    print(f"Acuity distribution: {cohort['esi_acuity'].value_counts().sort_index().to_dict()}")

    return cohort

In [25]:
# sample 30 patients and save
master_unique = (
    master
    .sort_values("ed_arrival_time", ascending=False)
    .drop_duplicates(subset="subject_id", keep="first")
    .copy()
)

sample_30 = select_balanced_cohort(master, n=10, seed=42)
sample_30.to_csv("/Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/mimic/selected_patients.csv", index=False)
print(f"\nSaved: /Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/mimic/selected_patients.csv")
sample_30.head(10)

Eligible rows (no NaN in triage features): 356,684 / 404,461
Unique patients after dedup: 180,057

Available patients per acuity level:
  ESI 1:  5,600  (need 2) [OK]
  ESI 2: 60,280  (need 2) [OK]
  ESI 3: 98,919  (need 2) [OK]
  ESI 4: 14,726  (need 2) [OK]
  ESI 5:    532  (need 2) [OK]
  Sampled 2 from ESI 1
  Sampled 2 from ESI 2
  Sampled 2 from ESI 3
  Sampled 2 from ESI 4
  Sampled 2 from ESI 5

Final cohort: 10 patients
Acuity distribution: {1: 2, 2: 2, 3: 2, 4: 2, 5: 2}

Saved: /Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/mimic/selected_patients.csv


,subject_id,hadm_id,stay_id,age,patient_gender,patient_race,arrival_mode,triage_temperature_f,triage_heart_rate_bpm,triage_resp_rate,triage_o2_saturation_pct,triage_systolic_bp,triage_diastolic_bp,triage_pain_score,chief_complaint,current_medications,medication_classes,num_current_medications,primary_language,marital_status,esi_acuity,ed_arrival_time,ed_disposition
0,17293255,NaN,33968379,57.0,M,WHITE,WALK IN,98.5,80.0,16.0,99.0,137.0,76.0,0.0,Suture removal,[levothyroxine],[Thyroid Hormones - Synthetic T4 (Thyroxine)],1,NaN,NaN,5,2136-11-28 10:47:00,HOME
1,12565578,NaN,36312788,69.0,F,BLACK/AFRICAN AMERICAN,WALK IN,98.0,75.0,16.0,100.0,131.0,58.0,0.0,Med refill,NaN,NaN,0,NaN,NaN,4,2179-10-13 20:43:00,HOME
2,14653207,28339257.0,31128182,93.0,F,BLACK/AFRICAN AMERICAN,AMBULANCE,98.0,89.0,20.0,94.0,135.0,72.0,10.0,"MVC, Transfer","[albuterol sulfate [ProAir HFA], azelastine, colchicine [Colcrys], Cyanocoba...","[Asthma/COPD Therapy - Beta 2-Adrenergic Agents, Inhaled, Short Acting, Nasa...",15,English,SINGLE,1,2196-05-01 17:55:00,ADMITTED
3,19808382,NaN,38962720,22.0,M,WHITE,WALK IN,97.8,74.0,18.0,98.0,156.0,95.0,0.0,Suture removal,NaN,NaN,0,NaN,NaN,5,2184-07-09 10:40:00,HOME
4,15638214,21969007.0,31512440,69.0,M,HISPANIC OR LATINO,AMBULANCE,98.6,85.0,20.0,99.0,111.0,72.0,0.0,DYSPNEA ON EXERTION,"[oxycodone, gabapentin, omeprazole, metoprolol succinate, OxyContin, prednis...","[Analgesic Opioid Agonists, Anticonvulsant - GABA Analogs, Gastric Acid Secr...",11,Spanish,MARRIED,2,2116-03-16 18:09:00,ADMITTED
5,11652499,26114984.0,33831108,53.0,M,WHITE,AMBULANCE,98.8,130.0,18.0,95.0,160.0,90.0,0.0,R/O SEPSIS,NaN,NaN,0,English,SINGLE,1,2156-08-13 21:20:00,ADMITTED
6,19352974,NaN,30771762,43.0,M,BLACK/AFRICAN AMERICAN,WALK IN,97.1,72.0,20.0,100.0,173.0,100.0,4.0,ABNL XRAY,[lisinopril],[ACE Inhibitors],1,NaN,NaN,3,2122-03-12 16:47:00,HOME
7,18263872,21475670.0,30366890,50.0,M,WHITE,AMBULANCE,97.4,90.0,18.0,97.0,132.0,98.0,9.0,Abd pain,"[Aveeno Moisturizing, blood sugar diagnostic [One Touch Ultra Test], bupropi...","[Dermatological - Emollients, Medical Supplies and DME - Blood Glucose Tests...",29,English,DIVORCED,2,2201-02-03 14:28:00,ADMITTED
8,16716170,NaN,34056271,35.0,M,HISPANIC/LATINO - DOMINICAN,WALK IN,98.3,91.0,18.0,100.0,143.0,94.0,7.0,"Abd pain, Vomiting",NaN,NaN,0,NaN,NaN,3,2134-07-18 14:29:00,HOME
9,12028289,NaN,38635725,25.0,M,OTHER,WALK IN,97.3,97.0,18.0,100.0,157.0,77.0,3.0,Dog bite,NaN,NaN,0,NaN,NaN,4,2177-04-09 23:34:00,HOME


In [26]:
sample_30 = select_balanced_cohort(master, n=30, seed=42)
sample_30.to_csv("/Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/pairwise_comparisons/mimic/selected_patients.csv", index=False)
print(f"\nSaved: /Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/pairwise_comparisons/mimic/selected_patients.csv")
sample_30.head(30)

Eligible rows (no NaN in triage features): 356,684 / 404,461
Unique patients after dedup: 180,057

Available patients per acuity level:
  ESI 1:  5,600  (need 6) [OK]
  ESI 2: 60,280  (need 6) [OK]
  ESI 3: 98,919  (need 6) [OK]
  ESI 4: 14,726  (need 6) [OK]
  ESI 5:    532  (need 6) [OK]
  Sampled 6 from ESI 1
  Sampled 6 from ESI 2
  Sampled 6 from ESI 3
  Sampled 6 from ESI 4
  Sampled 6 from ESI 5

Final cohort: 30 patients
Acuity distribution: {1: 6, 2: 6, 3: 6, 4: 6, 5: 6}

Saved: /Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/pairwise_comparisons/mimic/selected_patients.csv


,subject_id,hadm_id,stay_id,age,patient_gender,patient_race,arrival_mode,triage_temperature_f,triage_heart_rate_bpm,triage_resp_rate,triage_o2_saturation_pct,triage_systolic_bp,triage_diastolic_bp,triage_pain_score,chief_complaint,current_medications,medication_classes,num_current_medications,primary_language,marital_status,esi_acuity,ed_arrival_time,ed_disposition
0,18000475,NaN,36697220,23.0,F,OTHER,WALK IN,97.9,74.0,18.0,99.0,133.0,81.0,0.0,Rabies Vaccine,NaN,NaN,0,NaN,NaN,4,2148-08-04 20:25:00,HOME
1,15314576,NaN,31878427,43.0,M,WHITE,WALK IN,97.8,69.0,18.0,97.0,123.0,73.0,9.0,Neck pain,"[albuterol sulfate, Prilosec OTC]","[Asthma/COPD Therapy - Beta 2-Adrenergic Agents, Inhaled, Short Acting, Gast...",2,NaN,NaN,3,2189-04-09 19:00:00,HOME
2,13478335,22980159.0,37020835,70.0,M,WHITE,WALK IN,96.2,70.0,16.0,100.0,158.0,71.0,9.0,"Dizziness, Headache","[Lipitor, Multivitamin, Nexium, sucralfate, verapamil, amiodarone, Vitamin D...","[Antihyperlipidemic - HMG CoA Reductase Inhibitors (statins), Multivitamins,...",9,English,MARRIED,1,2112-05-18 21:41:00,ADMITTED
3,12028289,NaN,38635725,25.0,M,OTHER,WALK IN,97.3,97.0,18.0,100.0,157.0,77.0,3.0,Dog bite,NaN,NaN,0,NaN,NaN,4,2177-04-09 23:34:00,HOME
4,16716170,NaN,34056271,35.0,M,HISPANIC/LATINO - DOMINICAN,WALK IN,98.3,91.0,18.0,100.0,143.0,94.0,7.0,"Abd pain, Vomiting",NaN,NaN,0,NaN,NaN,3,2134-07-18 14:29:00,HOME
5,12694961,23789586.0,31265243,82.0,M,WHITE,WALK IN,98.8,80.0,16.0,97.0,112.0,67.0,3.0,? CONT'D DIARRHEA,"[aspirin, simvastatin]","[Salicylate Analgesics, Antihyperlipidemic - HMG CoA Reductase Inhibitors (s...",2,English,MARRIED,3,2125-06-05 11:20:00,ADMITTED
6,11652499,26114984.0,33831108,53.0,M,WHITE,AMBULANCE,98.8,130.0,18.0,95.0,160.0,90.0,0.0,R/O SEPSIS,NaN,NaN,0,English,SINGLE,1,2156-08-13 21:20:00,ADMITTED
7,15223188,21716019.0,36714392,50.0,F,BLACK/AFRICAN AMERICAN,WALK IN,99.0,142.0,22.0,97.0,143.0,103.0,10.0,ABDOMINAL PAIN,[atenolol],[Beta Blockers Cardiac Selective],1,English,SINGLE,1,2173-04-02 21:03:00,ADMITTED
8,19808382,NaN,38962720,22.0,M,WHITE,WALK IN,97.8,74.0,18.0,98.0,156.0,95.0,0.0,Suture removal,NaN,NaN,0,NaN,NaN,5,2184-07-09 10:40:00,HOME
9,16463594,NaN,34816329,49.0,F,ASIAN - SOUTH EAST ASIAN,WALK IN,97.8,77.0,16.0,100.0,126.0,84.0,8.0,Sinus pain,[cetirizine [Zyrtec]],"[Antihistamines - 2nd Generation, Antihistamines - 2nd Generation - Piperazi...",1,NaN,NaN,5,2185-02-20 10:20:00,HOME
